## Dataset Description: ACS Income (California)

The **ACS Income** dataset is derived from the US Census Bureau's American Community Survey (ACS). The task is to predict whether an individual's income is above $50,000.

### Columns:
- **AGEP**: Age
- **COW**: Class of worker (e.g., Private for-profit, Local government, Self-employed)
- **SCHL**: Education (e.g., High school diploma, Bachelor's degree)
- **MAR**: Marital status (e.g., Married, Divorced, Never married)
- **OCCP**: Occupation 
- **POBP**: Place of birth 
- **RELP**: Relationship to the reference person (e.g., Reference person, Husband/wife, Child)
- **WKHP**: Usual hours worked per week past 12 months
- **SEX**: Sex (1: Male, 2: Female)
- **RAC1P**: Recoded detailed race code (1: White, 2: Black or African American, etc.)
- **PINCP**: Total person's income (Target variable). In this dataset, it is a binary label: `1` if income > $50,000, `0` otherwise.


## 1. Data Downloading
Downloading the ACS Income data for California (2018).

In [1]:
import os
import pandas as pd
from folktables import ACSDataSource, ACSIncome
from sklearn.model_selection import train_test_split

DATASET_NAME = "acs-income-ca"
STATE = "CA"
YEAR = "2018"

OUTPUT_DIR = f"../../resources/datasets/{DATASET_NAME}"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Download Data
data_source = ACSDataSource(survey_year=YEAR, horizon='1-Year', survey='person')
acs_data = data_source.get_data(states=[STATE], download=True)

# Extract Features, Label, and Group
features, label, group = ACSIncome.df_to_pandas(acs_data)

# Combine into one DataFrame
df = pd.concat([features, label, group], axis=1)

# Remove duplicate columns if any
df = df.loc[:, ~df.columns.duplicated()]

print(f"Data shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Split into Train/Test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Save
train_path = os.path.join(OUTPUT_DIR, "train.csv")
test_path = os.path.join(OUTPUT_DIR, "test.csv")

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Saved train.csv to {train_path}")
print(f"Saved test.csv to {test_path}")

Data shape: (195665, 11)
Columns: ['AGEP', 'COW', 'SCHL', 'MAR', 'OCCP', 'POBP', 'RELP', 'WKHP', 'SEX', 'RAC1P', 'PINCP']
Saved train.csv to ../../resources/datasets/acs-income-ca/train.csv
Saved test.csv to ../../resources/datasets/acs-income-ca/test.csv
Saved train.csv to ../../resources/datasets/acs-income-ca/train.csv
Saved test.csv to ../../resources/datasets/acs-income-ca/test.csv


## 2. Baseline Model (LightGBM)
Training a LightGBM model using the best hyperparameters found by AIDE.


In [3]:
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

# Load data
train_path = "../../resources/datasets/acs-income-ca/train.csv"
test_path = "../../resources/datasets/acs-income-ca/test.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

# Features and target
X = train.drop("PINCP", axis=1)
y = train["PINCP"].astype(int)

X_test = test.drop("PINCP", axis=1)
y_test = test["PINCP"].astype(int)

# Categorical columns
# Ensure these columns exist in the dataset
categorical_cols = ["COW", "MAR", "RAC1P", "RELP", "SCHL", "SEX"]

# Cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Hyperparameter grid
learning_rates = [0.05, 0.1]
num_leaves_list = [31, 63, 127]
feature_fractions = [0.8, 1.0]
bagging_fractions = [0.8, 1.0]

best_acc = 0.0
best_params = {}
best_iter = 0

print("Starting Hyperparameter Search...")

for lr in learning_rates:
    for nl in num_leaves_list:
        for ff in feature_fractions:
            for bf in bagging_fractions:
                fold_accuracies = []
                fold_iters = []
                for train_idx, val_idx in skf.split(X, y):
                    X_tr, X_val = X.iloc[train_idx].reset_index(drop=True), X.iloc[val_idx].reset_index(drop=True)
                    y_tr, y_val = y.iloc[train_idx].reset_index(drop=True), y.iloc[val_idx].reset_index(drop=True)

                    # Ordinal encoding
                    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
                    X_tr_enc = X_tr.copy()
                    X_val_enc = X_val.copy()
                    X_tr_enc[categorical_cols] = encoder.fit_transform(X_tr[categorical_cols])
                    X_val_enc[categorical_cols] = encoder.transform(X_val[categorical_cols])

                    # Train model with early stopping and subsampling
                    clf = lgb.LGBMClassifier(
                        n_estimators=10000,
                        learning_rate=lr,
                        num_leaves=nl,
                        feature_fraction=ff,
                        bagging_fraction=bf,
                        bagging_freq=5,
                        random_state=42,
                        verbose=-1
                    )
                    clf.fit(
                        X_tr_enc,
                        y_tr,
                        eval_set=[(X_val_enc, y_val)],
                        eval_metric="binary_error",
                        categorical_feature=categorical_cols,
                        callbacks=[
                            lgb.early_stopping(stopping_rounds=50, verbose=False),
                        ]
                    )

                    preds = clf.predict(X_val_enc)
                    fold_accuracies.append(accuracy_score(y_val, preds))
                    fold_iters.append(clf.best_iteration_)

                mean_acc = np.mean(fold_accuracies)
                mean_iter = int(np.mean(fold_iters))
                print(f"Params lr={lr}, nl={nl}, ff={ff}, bf={bf} -> CV Acc = {mean_acc:.4f}, Iter = {mean_iter}")
                
                if mean_acc > best_acc:
                    best_acc = mean_acc
                    best_params = {
                        "learning_rate": lr,
                        "num_leaves": nl,
                        "feature_fraction": ff,
                        "bagging_fraction": bf,
                    }
                    best_iter = mean_iter

print(f"Best CV Accuracy: {best_acc:.4f} with params {best_params} and n_estimators={best_iter}")

# Retrain on full data with best params
print("Retraining on full training set...")
encoder_full = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_full_enc = X.copy()
X_full_enc[categorical_cols] = encoder_full.fit_transform(X_full_enc[categorical_cols])

X_test_enc = X_test.copy()
X_test_enc[categorical_cols] = encoder_full.transform(X_test_enc[categorical_cols])

final_clf = lgb.LGBMClassifier(
    n_estimators=best_iter,
    learning_rate=best_params["learning_rate"],
    num_leaves=best_params["num_leaves"],
    feature_fraction=best_params["feature_fraction"],
    bagging_fraction=best_params["bagging_fraction"],
    bagging_freq=5,
    random_state=42,
    verbose=-1
)
final_clf.fit(X_full_enc, y, categorical_feature=categorical_cols)

# Predict on test set
test_preds = final_clf.predict(X_test_enc).astype(int)
test_acc = accuracy_score(y_test, test_preds)
print(f"Test Accuracy: {test_acc:.4f}")

Starting Hyperparameter Search...
Params lr=0.05, nl=31, ff=0.8, bf=0.8 -> CV Acc = 0.8263, Iter = 304
Params lr=0.05, nl=31, ff=0.8, bf=0.8 -> CV Acc = 0.8263, Iter = 304
Params lr=0.05, nl=31, ff=0.8, bf=1.0 -> CV Acc = 0.8269, Iter = 392
Params lr=0.05, nl=31, ff=0.8, bf=1.0 -> CV Acc = 0.8269, Iter = 392
Params lr=0.05, nl=31, ff=1.0, bf=0.8 -> CV Acc = 0.8260, Iter = 286
Params lr=0.05, nl=31, ff=1.0, bf=0.8 -> CV Acc = 0.8260, Iter = 286
Params lr=0.05, nl=31, ff=1.0, bf=1.0 -> CV Acc = 0.8267, Iter = 378
Params lr=0.05, nl=31, ff=1.0, bf=1.0 -> CV Acc = 0.8267, Iter = 378
Params lr=0.05, nl=63, ff=0.8, bf=0.8 -> CV Acc = 0.8275, Iter = 248
Params lr=0.05, nl=63, ff=0.8, bf=0.8 -> CV Acc = 0.8275, Iter = 248
Params lr=0.05, nl=63, ff=0.8, bf=1.0 -> CV Acc = 0.8277, Iter = 303
Params lr=0.05, nl=63, ff=0.8, bf=1.0 -> CV Acc = 0.8277, Iter = 303
Params lr=0.05, nl=63, ff=1.0, bf=0.8 -> CV Acc = 0.8273, Iter = 256
Params lr=0.05, nl=63, ff=1.0, bf=0.8 -> CV Acc = 0.8273, Iter = 256


## 3. Baseline Fairness Evaluation
Evaluating the fairness of the baseline model with respect to Race (`RAC1P`).
We calculate:
- **Demographic Parity Difference**: The difference between the largest and smallest selection rates (percentage of positive predictions) across groups.
- **Equal Opportunity Difference**: The difference between the largest and smallest True Positive Rates across groups.

In [6]:
# Fairness Evaluation for Baseline
from sklearn.metrics import confusion_matrix

def calculate_fairness_metrics(y_true, y_pred, sensitive_features):
    """
    Calculates accuracy, demographic parity difference, and equal opportunity difference.
    """
    df = pd.DataFrame({
        'y_true': y_true,
        'y_pred': y_pred,
        'group': sensitive_features
    })
    
    # Accuracy
    acc = accuracy_score(y_true, y_pred)
    
    # Selection Rate (for Demographic Parity)
    selection_rates = df.groupby('group')['y_pred'].mean()
    dp_diff = selection_rates.max() - selection_rates.min()
    
    # True Positive Rate (for Equal Opportunity)
    # TPR = TP / Actual Positives
    # Filter for actual positives
    pos_df = df[df['y_true'] == 1]
    tpr_rates = pos_df.groupby('group')['y_pred'].mean()
    eo_diff = tpr_rates.max() - tpr_rates.min()
    
    return {
        "Accuracy": acc,
        "DP Difference": dp_diff,
        "EO Difference": eo_diff,
        "Selection Rates": selection_rates.to_dict(),
        "TPR Rates": tpr_rates.to_dict()
    }

# Run evaluation
metrics = calculate_fairness_metrics(y_test, test_preds, test["RAC1P"])

print("Baseline Model Fairness Metrics")
print(f"Accuracy: {metrics['Accuracy']:.4f}")
print(f"Demographic Parity Difference: {metrics['DP Difference']:.4f}")
print(f"Equal Opportunity Difference: {metrics['EO Difference']:.4f}")

print("\nSelection Rates by Group (RAC1P):")
for group, rate in metrics['Selection Rates'].items():
    print(f"Group {group}: {rate:.4f}")

print("\nTPR Rates by Group (RAC1P):")
for group, rate in metrics['TPR Rates'].items():
    print(f"Group {group}: {rate:.4f}")

Baseline Model Fairness Metrics
Accuracy: 0.8281
Demographic Parity Difference: 0.5399
Equal Opportunity Difference: 0.5595

Selection Rates by Group (RAC1P):
Group 1.0: 0.4528
Group 2.0: 0.3222
Group 3.0: 0.2490
Group 4.0: 0.6667
Group 5.0: 0.2024
Group 6.0: 0.4907
Group 7.0: 0.3008
Group 8.0: 0.1267
Group 9.0: 0.3625

TPR Rates by Group (RAC1P):
Group 1.0: 0.8157
Group 2.0: 0.6720
Group 3.0: 0.6957
Group 4.0: 1.0000
Group 5.0: 0.5000
Group 6.0: 0.8242
Group 7.0: 0.7778
Group 8.0: 0.4405
Group 9.0: 0.7692


## 4. Fairness-Prompted Model (Unawareness + Custom Thresholds)
This approach combines two strategies:
1.  **Fairness through Unawareness**: We explicitly remove the sensitive attribute (`RAC1P`) from the training features.
2.  **Group-Specific Thresholding**: During inference, we use different classification thresholds for each racial group to ensure that the selection rate (percentage of positive predictions) matches the overall target rate. This helps enforce Demographic Parity.


In [5]:
# Features and target
features = ["AGEP", "COW", "MAR", "OCCP", "POBP", "RELP", "SCHL", "SEX", "WKHP"]
# Ensure features exist
features = [f for f in features if f in train.columns]

X = train[features]
y = train["PINCP"].astype(int)
race = train["RAC1P"].astype(int)
X_test = test[features]
race_test = test["RAC1P"].astype(int)

# Prepare CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define hyperparameter grid
param_grid = [
    {"n_estimators": n, "num_leaves": nl, "learning_rate": lr, "max_depth": md}
    for n in [100, 200]
    for nl in [31, 63]
    for lr in [0.05, 0.1]
    for md in [-1, 7]
]

best_acc = 0
best_dp = None
best_params = None

print("Starting Fairness-Prompted Hyperparameter Search...")

# Grid search
for params in param_grid:
    accs, dp_diffs = [], []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        race_tr, race_val = race.iloc[tr_idx], race.iloc[val_idx]
        
        clf = lgb.LGBMClassifier(random_state=42, verbose=-1, **params)
        clf.fit(X_tr, y_tr)
        
        p_tr = clf.predict_proba(X_tr)[:, 1]
        p_val = clf.predict_proba(X_val)[:, 1]
        
        p_rate = y_tr.mean()
        thresholds = {}
        for r in np.unique(race_tr):
            probs_r = p_tr[race_tr == r]
            thresholds[r] = (
                np.quantile(probs_r, 1 - p_rate) if len(probs_r) > 0 else 0.5
            )
            
        y_pred = np.array(
            [1 if p_val[i] >= thresholds[r] else 0 for i, r in enumerate(race_val)]
        )
        
        accs.append(accuracy_score(y_val, y_pred))
        
        rates = [
            y_pred[race_val == r].mean()
            for r in np.unique(race_val)
            if (race_val == r).sum() > 0
        ]
        dp_diffs.append(max(rates) - min(rates))
        
    mean_acc = np.mean(accs)
    mean_dp = np.mean(dp_diffs)
    
    if mean_acc > best_acc:
        best_acc, best_dp, best_params = mean_acc, mean_dp, params

print(f"Best params: {best_params}")
print(f"Best CV Accuracy: {best_acc:.4f}")
print(f"Corresponding DP Difference: {best_dp:.4f}")

# Retrain on full data with best params
print("Retraining Fairness Model on full data...")
clf_full = lgb.LGBMClassifier(random_state=42, verbose=-1, **best_params)
clf_full.fit(X, y)

p_full = clf_full.predict_proba(X)[:, 1]
p_rate_full = y.mean()
thresholds_full = {}
for r in np.unique(race):
    probs_r = p_full[race == r]
    thresholds_full[r] = (
        np.quantile(probs_r, 1 - p_rate_full) if len(probs_r) > 0 else 0.5
    )

print("Group-Specific Thresholds:")
print(thresholds_full)

# Predict on test
p_test = clf_full.predict_proba(X_test)[:, 1]
y_test_pred = np.array(
    [1 if p_test[i] >= thresholds_full[r] else 0 for i, r in enumerate(race_test)]
)

Starting Fairness-Prompted Hyperparameter Search...
Best params: {'n_estimators': 200, 'num_leaves': 63, 'learning_rate': 0.1, 'max_depth': 7}
Best CV Accuracy: 0.8075
Corresponding DP Difference: 0.4439
Retraining Fairness Model on full data...
Best params: {'n_estimators': 200, 'num_leaves': 63, 'learning_rate': 0.1, 'max_depth': 7}
Best CV Accuracy: 0.8075
Corresponding DP Difference: 0.4439
Retraining Fairness Model on full data...
Group-Specific Thresholds:
{1: 0.5661247456171982, 2: 0.43237186537412386, 3: 0.32804157296735886, 4: 0.32852287175578554, 5: 0.18700185188739318, 6: 0.6378913958631608, 7: 0.3635379284749798, 8: 0.14535215432306092, 9: 0.38964361200225384}
Group-Specific Thresholds:
{1: 0.5661247456171982, 2: 0.43237186537412386, 3: 0.32804157296735886, 4: 0.32852287175578554, 5: 0.18700185188739318, 6: 0.6378913958631608, 7: 0.3635379284749798, 8: 0.14535215432306092, 9: 0.38964361200225384}


In [8]:
# Evaluate Fairness-Prompted Model
metrics_fair = calculate_fairness_metrics(y_test, y_test_pred, race_test)

print("\nFairness-Prompted Model Metrics:")
print(f"Accuracy: {metrics_fair['Accuracy']:.4f}")
print(f"Demographic Parity Difference: {metrics_fair['DP Difference']:.4f}")
print(f"Equal Opportunity Difference: {metrics_fair['EO Difference']:.4f}")

print("\nSelection Rates by Group (RAC1P):")
for group, rate in metrics_fair['Selection Rates'].items():
    print(f"Group {group}: {rate:.4f}")

print("\nTPR Rates by Group (RAC1P):")
for group, rate in metrics_fair['TPR Rates'].items():
    print(f"Group {group}: {rate:.4f}")


Fairness-Prompted Model Metrics:
Accuracy: 0.8114
Demographic Parity Difference: 0.2737
Equal Opportunity Difference: 0.2766

Selection Rates by Group (RAC1P):
Group 1: 0.4131
Group 2: 0.4178
Group 3: 0.3930
Group 4: 0.6667
Group 5: 0.4286
Group 6: 0.4054
Group 7: 0.4390
Group 8: 0.4000
Group 9: 0.4211

TPR Rates by Group (RAC1P):
Group 1: 0.7685
Group 2: 0.7884
Group 3: 0.8261
Group 4: 1.0000
Group 5: 0.9091
Group 6: 0.7234
Group 7: 0.9167
Group 8: 0.8741
Group 9: 0.8494


## 5. Baseline Model Post-processed (ThresholdOptimizer)
Using `fairlearn`'s `ThresholdOptimizer` to post-process the baseline model's predictions to achieve Demographic Parity.
This method requires splitting the training data into a training set (for the model) and a calibration set (for the optimizer).

In [9]:
from fairlearn.postprocessing import ThresholdOptimizer
from sklearn.model_selection import train_test_split

# Configuration
TARGET_COL = "PINCP"
SENSITIVE_COL = "RAC1P"
CATEGORICAL_COLS = ["COW", "MAR", "RAC1P", "RELP", "SCHL", "SEX"]

# Use existing train/test dataframes
# train and test are already loaded in the notebook

X = train.drop(TARGET_COL, axis=1)
y = train[TARGET_COL].astype(int)

X_test = test.drop(TARGET_COL, axis=1)
y_test = test[TARGET_COL].astype(int)

# Create a stratification key combining Target and Sensitive Attribute
stratify_col = y.astype(str) + "_" + X[SENSITIVE_COL].astype(str)

# Split train into train_main (for model) and train_calib (for threshold optimizer)
print("Splitting data into Main (75%) and Calibration (25%)...")
X_train_main, X_train_calib, y_train_main, y_train_calib = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=stratify_col
)

# Extract sensitive features
A_train_calib = X_train_calib[SENSITIVE_COL]
A_test = X_test[SENSITIVE_COL]

# Encoding
print("Encoding features...")
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X_train_main_enc = X_train_main.copy()
X_train_main_enc[CATEGORICAL_COLS] = encoder.fit_transform(X_train_main[CATEGORICAL_COLS])

X_train_calib_enc = X_train_calib.copy()
X_train_calib_enc[CATEGORICAL_COLS] = encoder.transform(X_train_calib[CATEGORICAL_COLS])

X_test_enc = X_test.copy()
X_test_enc[CATEGORICAL_COLS] = encoder.transform(X_test[CATEGORICAL_COLS])

# Train Baseline Model (LightGBM)
print("Training baseline LightGBM model on Main split...")
lgbm_baseline = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1
)

lgbm_baseline.fit(
    X_train_main_enc, 
    y_train_main, 
    categorical_feature=CATEGORICAL_COLS
)

# Post-processing with ThresholdOptimizer
print("Running ThresholdOptimizer (Demographic Parity)...")
optimizer = ThresholdOptimizer(
    estimator=lgbm_baseline,
    constraints="demographic_parity",
    predict_method='predict_proba',
    prefit=True
)

# Fit optimizer on calibration set
optimizer.fit(X_train_calib_enc, y_train_calib, sensitive_features=A_train_calib)

# Predict on test set
print("Predicting with Post-processed Fair Model...")
y_pred_postprocessed = optimizer.predict(X_test_enc, sensitive_features=A_test)

Splitting data into Main (75%) and Calibration (25%)...
Encoding features...
Training baseline LightGBM model on Main split...
Training baseline LightGBM model on Main split...
Running ThresholdOptimizer (Demographic Parity)...
Running ThresholdOptimizer (Demographic Parity)...
Predicting with Post-processed Fair Model...
Predicting with Post-processed Fair Model...


In [10]:
# Evaluate Post-processed Model
metrics_post = calculate_fairness_metrics(y_test, y_pred_postprocessed, A_test)

print("\nBaseline Model Post-processed Metrics:")
print(f"Accuracy: {metrics_post['Accuracy']:.4f}")
print(f"Demographic Parity Difference: {metrics_post['DP Difference']:.4f}")
print(f"Equal Opportunity Difference: {metrics_post['EO Difference']:.4f}")

print("\nSelection Rates by Group (RAC1P):")
for group, rate in metrics_post['Selection Rates'].items():
    print(f"Group {group}: {rate:.4f}")

print("\nTPR Rates by Group (RAC1P):")
for group, rate in metrics_post['TPR Rates'].items():
    print(f"Group {group}: {rate:.4f}")


Baseline Model Post-processed Metrics:
Accuracy: 0.8111
Demographic Parity Difference: 0.1090
Equal Opportunity Difference: 0.3776

Selection Rates by Group (RAC1P):
Group 1.0: 0.4209
Group 2.0: 0.4423
Group 3.0: 0.3696
Group 4.0: 0.3333
Group 5.0: 0.4405
Group 6.0: 0.4222
Group 7.0: 0.3821
Group 8.0: 0.4100
Group 9.0: 0.4199

TPR Rates by Group (RAC1P):
Group 1.0: 0.7776
Group 2.0: 0.8113
Group 3.0: 0.8116
Group 4.0: 0.5000
Group 5.0: 0.8636
Group 6.0: 0.7452
Group 7.0: 0.8611
Group 8.0: 0.8776
Group 9.0: 0.8511


## 6. Fairness-Prompted Model Post-processed (ThresholdOptimizer)
This section applies `fairlearn`'s `ThresholdOptimizer` to the "Fairness-Prompted" model (which uses Unawareness). This allows us to compare the manual thresholding heuristic from Section 4 with the optimized thresholding from `fairlearn`.

In [11]:
# Configuration
TARGET_COL = "PINCP"
SENSITIVE_COL = "RAC1P"
# Features used by the "Fairness Prompted" AIDE solution (Unawareness)
FEATURES_FAIR = ["AGEP", "COW", "MAR", "OCCP", "POBP", "RELP", "SCHL", "SEX", "WKHP"]
CATEGORICAL_COLS_FAIR = ["COW", "MAR", "RELP", "SCHL", "SEX"] # Removed RAC1P

# Prepare X and y
X_fair_pp = train[FEATURES_FAIR]
y_fair_pp = train[TARGET_COL].astype(int)
A_fair_pp = train[SENSITIVE_COL]

X_test_fair_pp = test[FEATURES_FAIR]
y_test_fair_pp = test[TARGET_COL].astype(int)
A_test_fair_pp = test[SENSITIVE_COL]

# Stratification Key
stratify_col = y_fair_pp.astype(str) + "_" + A_fair_pp.astype(str)

# Combine X and A for splitting to keep them aligned
X_with_A = X_fair_pp.copy()
X_with_A[SENSITIVE_COL] = A_fair_pp

print("Splitting data into Main (75%) and Calibration (25%)...")
X_train_main_full, X_train_calib_full, y_train_main, y_train_calib = train_test_split(
    X_with_A, y_fair_pp, test_size=0.25, random_state=42, stratify=stratify_col
)

# Separate A back out
A_train_calib = X_train_calib_full[SENSITIVE_COL]
X_train_main = X_train_main_full.drop(SENSITIVE_COL, axis=1)
X_train_calib = X_train_calib_full.drop(SENSITIVE_COL, axis=1)

# Encoding
print("Encoding features...")
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X_train_main_enc = X_train_main.copy()
X_train_main_enc[CATEGORICAL_COLS_FAIR] = encoder.fit_transform(X_train_main[CATEGORICAL_COLS_FAIR])

X_train_calib_enc = X_train_calib.copy()
X_train_calib_enc[CATEGORICAL_COLS_FAIR] = encoder.transform(X_train_calib[CATEGORICAL_COLS_FAIR])

X_test_enc = X_test_fair_pp.copy()
X_test_enc[CATEGORICAL_COLS_FAIR] = encoder.transform(X_test_fair_pp[CATEGORICAL_COLS_FAIR])

# Train Model (Mimicking Fairness Prompted Params)
print("Training LightGBM (Fairness Config)...")
# Params from AIDE's best_solution.py
lgbm_fair_pp = lgb.LGBMClassifier(
    n_estimators=200,
    num_leaves=31,
    learning_rate=0.05,
    max_depth=7,
    random_state=42,
    verbose=-1
)

lgbm_fair_pp.fit(
    X_train_main_enc, 
    y_train_main, 
    categorical_feature=CATEGORICAL_COLS_FAIR
)

# Post-processing with ThresholdOptimizer
print("Running ThresholdOptimizer on Unawareness Model...")
optimizer_fair = ThresholdOptimizer(
    estimator=lgbm_fair_pp,
    constraints="demographic_parity",
    predict_method='predict_proba',
    prefit=True
)

optimizer_fair.fit(X_train_calib_enc, y_train_calib, sensitive_features=A_train_calib)

print("Predicting with Fairlearn...")
y_pred_fair_pp = optimizer_fair.predict(X_test_enc, sensitive_features=A_test_fair_pp)

Splitting data into Main (75%) and Calibration (25%)...
Encoding features...
Training LightGBM (Fairness Config)...
Training LightGBM (Fairness Config)...
Running ThresholdOptimizer on Unawareness Model...
Running ThresholdOptimizer on Unawareness Model...
Predicting with Fairlearn...
Predicting with Fairlearn...


In [12]:
# Evaluate Fairness-Prompted Post-processed Model
metrics_fair_pp = calculate_fairness_metrics(y_test_fair_pp, y_pred_fair_pp, A_test_fair_pp)

print("\nFairness-Prompted Model Post-processed Metrics:")
print(f"Accuracy: {metrics_fair_pp['Accuracy']:.4f}")
print(f"Demographic Parity Difference: {metrics_fair_pp['DP Difference']:.4f}")
print(f"Equal Opportunity Difference: {metrics_fair_pp['EO Difference']:.4f}")

print("\nSelection Rates by Group (RAC1P):")
for group, rate in metrics_fair_pp['Selection Rates'].items():
    print(f"Group {group}: {rate:.4f}")

print("\nTPR Rates by Group (RAC1P):")
for group, rate in metrics_fair_pp['TPR Rates'].items():
    print(f"Group {group}: {rate:.4f}")


Fairness-Prompted Model Post-processed Metrics:
Accuracy: 0.8073
Demographic Parity Difference: 0.4226
Equal Opportunity Difference: 0.8627

Selection Rates by Group (RAC1P):
Group 1.0: 0.4116
Group 2.0: 0.4226
Group 3.0: 0.3813
Group 4.0: 0.0000
Group 5.0: 0.3929
Group 6.0: 0.4109
Group 7.0: 0.3984
Group 8.0: 0.3913
Group 9.0: 0.4199

TPR Rates by Group (RAC1P):
Group 1.0: 0.7611
Group 2.0: 0.7848
Group 3.0: 0.8116
Group 4.0: 0.0000
Group 5.0: 0.8182
Group 6.0: 0.7259
Group 7.0: 0.8056
Group 8.0: 0.8627
Group 9.0: 0.8347


## 7. Data Bias Analysis
Analyzing the inherent bias in the training data by calculating the Demographic Parity Difference on the ground truth labels. This helps us understand if the dataset itself is biased.

In [13]:
from fairlearn.metrics import demographic_parity_difference
import pandas as pd
import os

def check_data_bias(df, target_col, protected_col):
    print(f"\n--- Checking Data Bias ---")
    
    if target_col not in df.columns or protected_col not in df.columns:
        print(f"Error: Columns not found. Target: {target_col}, Protected: {protected_col}")
        return

    y_true = df[target_col].values
    sensitive_features = df[protected_col].values
    
    # Ensure binary
    y_true_binary = y_true.astype(int)
    
    # Calculate Demographic Parity on Ground Truth
    # We pass y_true_binary as the "predictions" to see the bias in the labels themselves
    bias_score = demographic_parity_difference(
        y_true_binary, 
        y_true_binary, 
        sensitive_features=sensitive_features
    )
    
    print(f"Ground Truth Demographic Parity Difference: {bias_score:.4f}")
    
    # Also print base rates for context
    print("\nBase Rates (Selection Rate) per Group:")
    groups = sorted(list(set(sensitive_features)))
    for g in groups:
        mask = sensitive_features == g
        rate = y_true_binary[mask].mean()
        count = mask.sum()
        print(f"  Group {g}: {rate:.4f} ({count} samples)")

# Ensure train is available
if 'train' not in locals():
    if 'train_df' in locals():
        print("Using 'train_df' as 'train'.")
        train = train_df
    else:
        # Try loading from disk
        train_path = "../../resources/datasets/acs-income-ca/train.csv"
        if os.path.exists(train_path):
            print(f"Loading data from {train_path}...")
            train = pd.read_csv(train_path)
        else:
            print(f"Error: 'train' variable not found and data file not found at {train_path}.")
            train = None

# Run bias check
if train is not None:
    check_data_bias(train, "PINCP", "RAC1P")
else:
    print("Skipping bias check due to missing data.")


--- Checking Data Bias ---
Ground Truth Demographic Parity Difference: 0.2861

Base Rates (Selection Rate) per Group:
  Group 1.0: 0.4434 (96813 samples)
  Group 2.0: 0.3463 (6884 samples)
  Group 3.0: 0.2874 (1037 samples)
  Group 4.0: 0.3000 (10 samples)
  Group 5.0: 0.2022 (366 samples)
  Group 6.0: 0.4803 (26029 samples)
  Group 7.0: 0.2996 (514 samples)
  Group 8.0: 0.1943 (18295 samples)
  Group 9.0: 0.3524 (6584 samples)
Ground Truth Demographic Parity Difference: 0.2861

Base Rates (Selection Rate) per Group:
  Group 1.0: 0.4434 (96813 samples)
  Group 2.0: 0.3463 (6884 samples)
  Group 3.0: 0.2874 (1037 samples)
  Group 4.0: 0.3000 (10 samples)
  Group 5.0: 0.2022 (366 samples)
  Group 6.0: 0.4803 (26029 samples)
  Group 7.0: 0.2996 (514 samples)
  Group 8.0: 0.1943 (18295 samples)
  Group 9.0: 0.3524 (6584 samples)
